# Dynamic A/B Test Interpreter
Feed it one or more CSVs and a plain-English problem statement. It:

1. **Ingests** the data — a single CSV with a group column, or multiple CSVs (one per group/variant)
2. **Infers the schema** — asks an LLM to read the columns, dtypes, sample rows, and your problem statement to figure out which column is the group/variant, which column is the metric to test, whether that metric is a proportion or continuous, and which group is the baseline
3. **Runs two-tailed pairwise tests** — every variant vs. the baseline: a two-proportion z-test for proportion metrics, Welch's t-test for continuous metrics
4. **Validates** the generated narrative against the actual numbers, to catch an LLM overstating significance
5. **Produces an aesthetic markdown report** — problem statement, methodology, a results table, per-comparison narrative, and an overall recommendation

Demonstrated on two example problems below: the Cookie Cats mobile-game retention test (2 groups, proportion metric) and a 3-way pricing experiment (base / 5% discount / 10% discount, continuous metric).

**To use with your own data:** point `csv_paths` at your file(s) and write your own `problem_statement` — the schema inspector figures out the rest. Swap `MockLLMClient` for `GroqLLMClient` (fill in the API call) to use a real model instead of the heuristic mock.

In [7]:
import os
from groq import Groq
API_KEY = os.environ.get("GROQ_API_KEY")
MODEL = os.environ.get("GROQ_MODEL")
client = Groq(api_key=API_KEY)

## 1. Core data structures
`ABTestResult` — one pairwise comparison (variant vs. baseline). `ExperimentSchema` — what gets inferred about the dataset: which column is the group, which is the metric, its type, and the baseline label.

In [8]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
import math


@dataclass
class ABTestResult:
    """One pairwise comparison: a variant group vs. the baseline group."""
    comparison_name: str
    baseline_label: str
    variant_label: str
    baseline_metric: float
    variant_metric: float
    baseline_n: int
    variant_n: int
    p_value: float
    metric_type: str  # "proportion" or "continuous"
    metric_name: str
    confidence_level: float = 0.95

    def lift(self) -> float:
        if self.baseline_metric == 0:
            return math.inf
        return (self.variant_metric - self.baseline_metric) / self.baseline_metric * 100

    def is_significant(self) -> bool:
        alpha = 1 - self.confidence_level
        return self.p_value < alpha

    def sample_size_flag(self, min_n: int = 100) -> bool:
        return self.baseline_n < min_n or self.variant_n < min_n

    def summary_stats(self) -> dict:
        return {
            "comparison_name": self.comparison_name,
            "metric_name": self.metric_name,
            "metric_type": self.metric_type,
            "baseline_label": self.baseline_label,
            "variant_label": self.variant_label,
            "baseline_metric": self.baseline_metric,
            "variant_metric": self.variant_metric,
            "lift_pct": round(self.lift(), 2),
            "p_value": self.p_value,
            "significant": self.is_significant(),
            "baseline_n": self.baseline_n,
            "variant_n": self.variant_n,
            "small_sample_warning": self.sample_size_flag(),
        }


@dataclass
class ExperimentSchema:
    """What the LLM (or a human) determines about the dataset's shape."""
    group_column: str
    metric_column: str
    metric_type: str  # "proportion" or "continuous"
    baseline_label: str
    groups: list[str] = field(default_factory=list)
    reasoning: str = ""


## 2. Data ingestion
Normalizes either a single CSV (already has a group column) or a dict of `{group_label: csv_path}` — one file per group — into one long DataFrame.

In [9]:
from __future__ import annotations
import pandas as pd


class ExperimentDataLoader:
    @staticmethod
    def load(csv_paths) -> pd.DataFrame:
        """
        csv_paths:
          - str: path to a single CSV that already contains a group column
                 (which group column it is gets figured out later by the
                 SchemaInspector).
          - dict[str, str]: {group_label: csv_path} — one file per group.
                 A '__group__' column is injected using the dict keys, so the
                 SchemaInspector doesn't need to guess a group column at all.
        """
        if isinstance(csv_paths, str):
            df = pd.read_csv(csv_paths)
            return df

        if isinstance(csv_paths, dict):
            frames = []
            for label, path in csv_paths.items():
                part = pd.read_csv(path)
                part = part.copy()
                part["__group__"] = label
                frames.append(part)
            return pd.concat(frames, ignore_index=True)

        raise TypeError("csv_paths must be a single path (str) or a dict of {group_label: path}")


## 3. LLM client
Thin wrapper so the pipeline doesn't care which provider is generating text. `MockLLMClient` here is a heuristic stand-in — it reads the *prompt text* (never the DataFrame directly) using keyword/dtype cues, the same information a real LLM would have. Swap in `GroqLLMClient` for the real thing; nothing else in the pipeline changes.

In [10]:
from __future__ import annotations
from abc import ABC, abstractmethod
import re


class LLMClient(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> str:
        ...


class GroqLLMClient(LLMClient):
    """Real client — uses the Groq API."""

    def __init__(self, model: str = MODEL, max_tokens: int = 500):
        self.model = model
        self.max_tokens = max_tokens
        self.client = client

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            max_tokens=self.max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

class MockLLMClient(LLMClient):
    """
    Fake client for local dev/testing without burning API calls or requiring
    a key. It uses light heuristics over the PROMPT TEXT ONLY (it never sees
    the DataFrame directly) to stand in for what a real LLM would infer from
    reading the same prompt. Swap in GroqLLMClient for the real thing —
    nothing else in the pipeline needs to change.
    """

    GROUP_KEYWORDS = ["group", "version", "variant", "arm", "cohort", "tier", "segment", "bucket"]
    PROPORTION_HINTS = ["bool", "0/1", "true/false", "binary", "retention", "conversion", "converted", "clicked", "purchased"]

    def generate(self, prompt: str) -> str:
        if "SCHEMA_INSPECTION_TASK" in prompt:
            return self._infer_schema(prompt)
        if "OVERALL_SUMMARY_TASK" in prompt:
            return self._overall_summary(prompt)
        return self._comparison_narrative(prompt)

    # -- schema inspection -------------------------------------------------

    def _infer_schema(self, prompt: str) -> str:
        columns = self._extract_columns(prompt)
        problem_statement = self._extract_field(prompt, "Problem statement")
        groups_hint = self._extract_field(prompt, "Candidate group labels")

        group_column = None
        for col, dtype in columns:
            if any(kw in col.lower() for kw in self.GROUP_KEYWORDS):
                group_column = col
                break
        if group_column is None:
            # falls back to the injected loader column for multi-CSV inputs
            for col, dtype in columns:
                if col == "__group__":
                    group_column = col
                    break

        metric_column = None
        metric_type = "continuous"
        ps_lower = (problem_statement or "").lower()
        id_like = ("userid", "id", "customer_id", "user_id", "__group__")
        candidate_cols = [c for c, d in columns if c not in (group_column,) and c.lower() not in id_like]

        # prefer a column whose name is echoed in the problem statement,
        # checking both the literal name and a space-separated variant
        for col in candidate_cols:
            col_lower = col.lower()
            col_spaced = col_lower.replace("_", " ")
            if col_lower in ps_lower or col_spaced in ps_lower:
                metric_column = col
                break

        if metric_column is None:
            # otherwise prefer anything that looks like a retention/conversion/outcome flag
            for col, dtype in columns:
                if col in candidate_cols and (any(h in col.lower() for h in self.PROPORTION_HINTS) or "bool" in dtype.lower()):
                    metric_column = col
                    metric_type = "proportion"
                    break

        if metric_column is None:
            # last resort: first numeric-looking column left
            for col, dtype in columns:
                if col in candidate_cols:
                    metric_column = col
                    break

        if metric_type != "proportion":
            for col, dtype in columns:
                if col == metric_column and ("bool" in dtype.lower() or any(h in col.lower() for h in self.PROPORTION_HINTS)):
                    metric_type = "proportion"

        groups = [g.strip().strip("'\"") for g in re.split(r"[,\[\]]", groups_hint) if g.strip()] if groups_hint else []
        baseline_label = None
        for g in groups:
            if any(k in g.lower() for k in ["base", "control", "30", "0%", "original"]):
                baseline_label = g
                break
        if baseline_label is None and groups:
            baseline_label = sorted(groups)[0]

        return (
            "{"
            f'"group_column": {self._j(group_column)}, '
            f'"metric_column": {self._j(metric_column)}, '
            f'"metric_type": {self._j(metric_type)}, '
            f'"baseline_label": {self._j(baseline_label)}'
            "}"
        )

    # -- narrative generation -----------------------------------------------

    def _comparison_narrative(self, prompt: str) -> str:
        significant = "significant: true" in prompt.lower()
        variant = self._extract_field(prompt, "Variant label") or "the variant"
        baseline = self._extract_field(prompt, "Baseline label") or "baseline"
        lift = (self._extract_field(prompt, "Lift") or "").rstrip("%")
        if significant:
            return (
                f"{variant} shows a statistically significant change vs. {baseline} "
                f"({lift}% relative difference). This result is unlikely to be due to chance "
                f"at the stated confidence level."
            )
        return (
            f"{variant} shows a {lift}% difference vs. {baseline}, but this is not "
            f"statistically significant at the stated confidence level — treat it as "
            f"inconclusive rather than a real effect."
        )

    def _overall_summary(self, prompt: str) -> str:
        n_sig = prompt.lower().count('"significant": true')
        n_total = prompt.lower().count('"comparison_name"')
        if n_sig == 0:
            return (
                "None of the tested variants produced a statistically significant "
                "difference from baseline. Recommend either extending the test for more "
                "data or concluding the variants tested do not meaningfully move the metric."
            )
        return (
            f"{n_sig} of {n_total} comparisons showed a statistically significant "
            "difference from baseline. Recommend reviewing those specific comparisons "
            "before rolling out any change broadly, and continuing to monitor "
            "non-significant variants rather than discarding them outright."
        )

    # -- helpers --------------------------------------------------------------

    @staticmethod
    def _j(value) -> str:
        if value is None:
            return "null"
        return '"' + str(value).replace('"', "'") + '"'

    @staticmethod
    def _extract_columns(prompt: str) -> list[tuple[str, str]]:
        m = re.search(r"Columns and dtypes:\s*(.*)", prompt)
        if not m:
            return []
        pairs = re.findall(r"(\w+)\s*\(([^)]+)\)", m.group(1))
        return pairs

    @staticmethod
    def _extract_field(prompt: str, field_name: str) -> str | None:
        m = re.search(rf"{re.escape(field_name)}:\s*(.+)", prompt)
        return m.group(1).strip() if m else None


## 4. Schema inspector
The "dynamic" core of the pipeline — builds a structured prompt from the dataframe's shape + your problem statement, and parses the LLM's JSON response into an `ExperimentSchema`.

In [11]:
from __future__ import annotations
import json
import re
import pandas as pd



class SchemaInspector:
    def __init__(self, client: LLMClient):
        self.client = client

    def inspect(self, df: pd.DataFrame, problem_statement: str) -> ExperimentSchema:
        prompt = self._build_prompt(df, problem_statement)
        raw = self.client.generate(prompt)
        parsed = self._parse(raw)

        group_column = parsed["group_column"]
        groups = sorted(df[group_column].dropna().unique().tolist()) if group_column else []
        baseline_label = parsed.get("baseline_label") or (groups[0] if groups else None)

        return ExperimentSchema(
            group_column=group_column,
            metric_column=parsed["metric_column"],
            metric_type=parsed["metric_type"],
            baseline_label=baseline_label,
            groups=groups,
        )

    def _build_prompt(self, df: pd.DataFrame, problem_statement: str) -> str:
        dtypes_str = ", ".join(f"{c}({str(t)})" for c, t in df.dtypes.items())
        sample_rows = df.head(3).to_dict(orient="records")

        # surface candidate group labels if an obvious group-like column exists,
        # to help identify the intended baseline
        candidate_group_col = None
        for c in df.columns:
            if any(kw in c.lower() for kw in ["group", "version", "variant", "arm", "cohort", "tier", "segment", "bucket"]):
                candidate_group_col = c
                break
        if candidate_group_col is None and "__group__" in df.columns:
            candidate_group_col = "__group__"
        groups_hint = sorted(df[candidate_group_col].dropna().unique().tolist()) if candidate_group_col else []

        return (
            "SCHEMA_INSPECTION_TASK\n"
            "You are analyzing a dataset for an A/B test. Identify:\n"
            "1) which column identifies the test arm/group a row belongs to,\n"
            "2) which column is the outcome metric to test,\n"
            "3) whether that metric is a proportion (binary/0-1, like converted or retained) "
            "or continuous (like revenue or session length),\n"
            "4) which group label should be treated as the baseline/control.\n\n"
            f"Problem statement: {problem_statement}\n"
            f"Columns and dtypes: {dtypes_str}\n"
            f"Sample rows: {sample_rows}\n"
            f"Candidate group labels: {groups_hint}\n\n"
            "Respond with ONLY valid JSON, no prose, no markdown fences, in this exact shape:\n"
            '{"group_column": "<col name>", "metric_column": "<col name>", '
            '"metric_type": "proportion" | "continuous", "baseline_label": "<group label>"}'
        )

    @staticmethod
    def _parse(raw: str) -> dict:
        cleaned = re.sub(r"```(json)?", "", raw).strip()
        return json.loads(cleaned)


## 5. Pairwise test runner
Runs a two-tailed test for every non-baseline group vs. baseline. Proportion metrics → two-proportion z-test. Continuous metrics → Welch's t-test (doesn't assume equal variance).

In [12]:
from __future__ import annotations
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest



class PairwiseTestRunner:
    def run(self, df: pd.DataFrame, schema: ExperimentSchema) -> list[ABTestResult]:
        group_col = schema.group_column
        metric_col = schema.metric_column
        baseline = schema.baseline_label

        baseline_series = df[df[group_col] == baseline][metric_col].dropna()
        results = []

        for label in schema.groups:
            if label == baseline:
                continue
            variant_series = df[df[group_col] == label][metric_col].dropna()

            if schema.metric_type == "proportion":
                p_value = self._proportion_test(baseline_series, variant_series)
            else:
                p_value = self._continuous_test(baseline_series, variant_series)

            results.append(ABTestResult(
                comparison_name=f"{label} vs {baseline}",
                baseline_label=baseline,
                variant_label=label,
                baseline_metric=float(baseline_series.mean()),
                variant_metric=float(variant_series.mean()),
                baseline_n=len(baseline_series),
                variant_n=len(variant_series),
                p_value=p_value,
                metric_type=schema.metric_type,
                metric_name=metric_col,
            ))

        return results

    @staticmethod
    def _proportion_test(baseline_series, variant_series) -> float:
        successes = [baseline_series.sum(), variant_series.sum()]
        n_obs = [len(baseline_series), len(variant_series)]
        _, p_value = proportions_ztest(successes, n_obs)
        return float(p_value)

    @staticmethod
    def _continuous_test(baseline_series, variant_series) -> float:
        _, p_value = stats.ttest_ind(baseline_series, variant_series, equal_var=False)
        return float(p_value)


## 6. Prompt templates
Turn a single comparison (or the full set) into natural-language narrative for the report.

In [13]:
from __future__ import annotations
import json
from abc import ABC, abstractmethod



class PromptTemplate(ABC):
    @abstractmethod
    def render(self, result: ABTestResult) -> str:
        ...

    def _stats_block(self, result: ABTestResult) -> str:
        s = result.summary_stats()
        return (
            f"Comparison name: {s['comparison_name']}\n"
            f"Metric: {s['metric_name']} ({s['metric_type']})\n"
            f"Baseline label: {s['baseline_label']}\n"
            f"Variant label: {s['variant_label']}\n"
            f"Baseline metric: {s['baseline_metric']:.4f}\n"
            f"Variant metric: {s['variant_metric']:.4f}\n"
            f"Lift: {s['lift_pct']}%\n"
            f"p-value: {s['p_value']:.4f}\n"
            f"Significant: {s['significant']}\n"
            f"Baseline n: {s['baseline_n']}, Variant n: {s['variant_n']}\n"
            f"Small sample warning: {s['small_sample_warning']}\n"
        )


class ComparisonNarrativePrompt(PromptTemplate):
    def render(self, result: ABTestResult) -> str:
        return (
            "Write a 1-2 sentence, statistically honest summary of this single "
            "pairwise A/B test comparison, for a report read by both analysts and "
            "business stakeholders. Never invent numbers not given below. If not "
            "significant, say so plainly and do not imply a decision should be made.\n\n"
            f"{self._stats_block(result)}\n"
            "Write the summary now."
        )


class OverallSummaryPrompt:
    def render(self, results: list[ABTestResult], problem_statement: str) -> str:
        blocks = "\n".join(json.dumps(r.summary_stats()) for r in results)
        return (
            "OVERALL_SUMMARY_TASK\n"
            "You are writing the closing recommendation section of an A/B test report "
            "covering multiple pairwise comparisons against a shared baseline.\n"
            f"Problem statement: {problem_statement}\n\n"
            f"All comparisons:\n{blocks}\n\n"
            "Write a 2-4 sentence overall recommendation. Be statistically honest — "
            "do not claim a comparison is significant if its data says otherwise. "
            "Never invent numbers not given above."
        )


## 7. Validator
Independently checks generated narrative against the actual stats — catches the LLM overstating significance.

In [14]:
from __future__ import annotations


class ResultValidator:
    SIGNIFICANT_CLAIM_WORDS = ["significant", "confirmed", "proven", "clear win", "definitely"]

    def validate(self, narrative: str, result: ABTestResult) -> list[str]:
        flags = []
        text_lower = narrative.lower()

        claims_significance = any(w in text_lower for w in self.SIGNIFICANT_CLAIM_WORDS)
        if claims_significance and not result.is_significant():
            flags.append(
                f"Narrative implies significance, but p-value ({result.p_value:.4f}) "
                f"does not meet the {result.confidence_level*100:.0f}% CI threshold."
            )

        if result.sample_size_flag() and "sample" not in text_lower and "small" not in text_lower:
            flags.append("Sample size is small but the narrative doesn't mention it as a caveat.")

        return flags


## 8. Interpreter
Composes one comparison + a prompt template + the LLM client + the validator.

In [15]:
from __future__ import annotations



class Interpreter:
    def __init__(self, result: ABTestResult, template: PromptTemplate, client: LLMClient):
        self.result = result
        self.template = template
        self.client = client
        self.validator = ResultValidator()

    def interpret(self) -> dict:
        prompt = self.template.render(self.result)
        narrative = self.client.generate(prompt)
        flags = self.validator.validate(narrative, self.result)
        return {"prompt": prompt, "narrative": narrative, "validation_flags": flags}


## 9. Markdown report builder
Assembles the final report: title, problem statement, methodology, results table, per-comparison narrative (with any validator flags), and overall recommendation.

In [16]:
from __future__ import annotations
from datetime import date



class MarkdownReportBuilder:
    def build(
        self,
        project_title: str,
        problem_statement: str,
        schema: ExperimentSchema,
        results: list[ABTestResult],
        narratives: dict[str, dict],
        overall_summary: str,
    ) -> str:
        lines = []
        lines.append(f"# {project_title}")
        lines.append("")
        lines.append(f"*Generated {date.today().isoformat()}*")
        lines.append("")
        lines.append("## Problem Statement")
        lines.append(problem_statement.strip())
        lines.append("")
        test_method = (
            "proportion — two-proportion z-test"
            if schema.metric_type == "proportion"
            else "continuous — Welch's t-test"
        )
        alpha = (1 - results[0].confidence_level) if results else 0.05
        groups_list = ", ".join(f"`{g}`" for g in schema.groups)
        lines.append("## Methodology")
        lines.append(
            f"- **Metric tested:** `{schema.metric_column}` ({test_method}), two-tailed\n"
            f"- **Baseline group:** `{schema.baseline_label}`\n"
            f"- **Groups compared:** {groups_list}\n"
            f"- **Significance threshold:** {alpha:.2f} (95% confidence level)"
        )
        lines.append("")
        lines.append("## Results at a Glance")
        lines.append("")
        lines.append("| Comparison | Baseline | Variant | Lift | p-value | Significant | Sample size |")
        lines.append("|---|---|---|---|---|---|---|")
        for r in results:
            s = r.summary_stats()
            sig_badge = "✅ Yes" if s["significant"] else "❌ No"
            size_badge = "⚠️ Small" if s["small_sample_warning"] else "OK"
            lines.append(
                f"| {s['comparison_name']} | {s['baseline_metric']:.4f} | {s['variant_metric']:.4f} | "
                f"{s['lift_pct']}% | {s['p_value']:.4f} | {sig_badge} | {size_badge} |"
            )
        lines.append("")
        lines.append("## Comparison Details")
        for r in results:
            n = narratives[r.comparison_name]
            lines.append("")
            lines.append(f"### {r.comparison_name}")
            lines.append(n["narrative"])
            if n["validation_flags"]:
                lines.append("")
                lines.append("> ⚠️ **Validator flags:**")
                for f in n["validation_flags"]:
                    lines.append(f"> - {f}")
        lines.append("")
        lines.append("## Overall Recommendation")
        lines.append(overall_summary)
        lines.append("")
        lines.append("---")
        lines.append("*Report generated by the Dynamic A/B Test Interpreter pipeline.*")

        return "\n".join(lines)


## 10. Pipeline
The top-level orchestrator: CSV(s) + problem statement in → markdown report out.

In [17]:
from __future__ import annotations



class ABTestPipeline:
    def __init__(self, client: LLMClient):
        self.client = client
        self.inspector = SchemaInspector(client)
        self.runner = PairwiseTestRunner()
        self.report_builder = MarkdownReportBuilder()

    def run(self, csv_paths, problem_statement: str, project_title: str) -> str:
        df = ExperimentDataLoader.load(csv_paths)
        schema = self.inspector.inspect(df, problem_statement)
        results = self.runner.run(df, schema)

        narratives = {}
        for r in results:
            narratives[r.comparison_name] = Interpreter(
                r, ComparisonNarrativePrompt(), self.client
            ).interpret()

        overall_prompt = OverallSummaryPrompt().render(results, problem_statement)
        overall_summary = self.client.generate(overall_prompt)

        return self.report_builder.build(
            project_title, problem_statement, schema, results, narratives, overall_summary
        )


## 11. Example 1 — Cookie Cats retention test
Single CSV with a `version` group column, proportion metric (`retention_7`).

*(No `cookie_cats.csv` in this folder? A small synthetic sample with the same shape is generated below so the notebook still runs end to end — swap in the real file from [Kaggle](https://www.kaggle.com/datasets/yufengsui/mobile-games-ab-testing) for real results.)*

In [18]:
import os
import numpy as np
import pandas as pd

if not os.path.exists("interpreter_input.csv"):
    rng = np.random.default_rng(42)
    n_control, n_treatment = 4470, 4548
    control = pd.DataFrame({
        "userid": range(n_control),
        "version": "gate_30",
        "sum_gamerounds": rng.poisson(50, n_control),
        "retention_1": rng.random(n_control) < 0.448,
        "retention_7": rng.random(n_control) < 0.190,
    })
    treatment = pd.DataFrame({
        "userid": range(n_control, n_control + n_treatment),
        "version": "gate_40",
        "sum_gamerounds": rng.poisson(50, n_treatment),
        "retention_1": rng.random(n_treatment) < 0.442,
        "retention_7": rng.random(n_treatment) < 0.182,
    })
    pd.concat([control, treatment], ignore_index=True).to_csv("interpreter_input.csv", index=False)
    print("Synthetic interpreter_input.csv generated.")
else:
    print("Using existing interpreter_input.csv.")

Using existing interpreter_input.csv.


In [20]:
client = MockLLMClient()  # swap for AnthropicLLMClient() to use a real model
pipeline = ABTestPipeline(client)

test_results = pipeline.run(
    csv_paths="interpreter_input.csv",
    problem_statement=(
        "We moved a progress gate in our mobile game from level 30 to level 40. "
        "Did this change affect player retention_7 (whether a player came back 7 days later)?"
    ),
    project_title="Cookie Cats: Gate Placement A/B Test",
)

with open("test_results.md", "w", encoding="utf-8") as f:
    f.write(test_results)

from IPython.display import Markdown, display
display(Markdown(test_results))

# Cookie Cats: Gate Placement A/B Test

*Generated 2026-08-27*

## Problem Statement
We moved a progress gate in our mobile game from level 30 to level 40. Did this change affect player retention_7 (whether a player came back 7 days later)?

## Methodology
- **Metric tested:** `retention_7` (proportion — two-proportion z-test), two-tailed
- **Baseline group:** `gate_30`
- **Groups compared:** `gate_30`, `gate_40`
- **Significance threshold:** 0.05 (95% confidence level)

## Results at a Glance

| Comparison | Baseline | Variant | Lift | p-value | Significant | Sample size |
|---|---|---|---|---|---|---|
| gate_40 vs gate_30 | 0.1902 | 0.1820 | -4.31% | 0.0016 | ✅ Yes | OK |

## Comparison Details

### gate_40 vs gate_30
gate_40 shows a statistically significant change vs. gate_30 (-4.31% relative difference). This result is unlikely to be due to chance at the stated confidence level.

## Overall Recommendation
1 of 1 comparisons showed a statistically significant difference from baseline. Recommend reviewing those specific comparisons before rolling out any change broadly, and continuing to monitor non-significant variants rather than discarding them outright.

---
*Report generated by the Dynamic A/B Test Interpreter pipeline.*

## 12. Example 2 — Pricing experiment (3 groups)
Three separate CSVs, one per price point, continuous metric (`revenue_per_customer`). This shows the pipeline handling more than two groups and a different metric type without any code changes — only the inputs differ.

In [21]:
if not (os.path.exists("price_base.csv") and os.path.exists("price_disc5.csv") and os.path.exists("price_disc10.csv")):
    rng = np.random.default_rng(42)
    n = 1200
    pd.DataFrame({
        "customer_id": range(n),
        "revenue_per_customer": rng.normal(42, 12, n).clip(0),
    }).to_csv("price_base.csv", index=False)
    pd.DataFrame({
        "customer_id": range(n, 2 * n),
        "revenue_per_customer": rng.normal(43.5, 12, n).clip(0),
    }).to_csv("price_disc5.csv", index=False)
    pd.DataFrame({
        "customer_id": range(2 * n, 3 * n),
        "revenue_per_customer": rng.normal(39, 12, n).clip(0),
    }).to_csv("price_disc10.csv", index=False)
    print("Synthetic pricing CSVs generated.")
else:
    print("Using existing pricing CSVs.")

Synthetic pricing CSVs generated.


In [23]:
pricing_report = pipeline.run(
    csv_paths={
        "base": "price_base.csv",
        "5pct_discount": "price_disc5.csv",
        "10pct_discount": "price_disc10.csv",
    },
    problem_statement=(
        "We are running a pricing experiment with three price points: base price, "
        "a 5% discount, and a 10% discount. We want to know whether either discount "
        "changes revenue_per_customer compared to the base price."
    ),
    project_title="Pricing Experiment: Base vs 5% vs 10% Discount",
)

with open("report_pricing.md", "w", encoding="utf-8") as f:
    f.write(pricing_report)

display(Markdown(pricing_report))

# Pricing Experiment: Base vs 5% vs 10% Discount

*Generated 2026-08-27*

## Problem Statement
We are running a pricing experiment with three price points: base price, a 5% discount, and a 10% discount. We want to know whether either discount changes revenue_per_customer compared to the base price.

## Methodology
- **Metric tested:** `revenue_per_customer` (continuous — Welch's t-test), two-tailed
- **Baseline group:** `base`
- **Groups compared:** `10pct_discount`, `5pct_discount`, `base`
- **Significance threshold:** 0.05 (95% confidence level)

## Results at a Glance

| Comparison | Baseline | Variant | Lift | p-value | Significant | Sample size |
|---|---|---|---|---|---|---|
| 10pct_discount vs base | 41.7346 | 39.4607 | -5.45% | 0.0000 | ✅ Yes | OK |
| 5pct_discount vs base | 41.7346 | 42.7456 | 2.42% | 0.0399 | ✅ Yes | OK |

## Comparison Details

### 10pct_discount vs base
10pct_discount shows a statistically significant change vs. base (-5.45% relative difference). This result is unlikely to be due to chance at the stated confidence level.

### 5pct_discount vs base
5pct_discount shows a statistically significant change vs. base (2.42% relative difference). This result is unlikely to be due to chance at the stated confidence level.

## Overall Recommendation
2 of 2 comparisons showed a statistically significant difference from baseline. Recommend reviewing those specific comparisons before rolling out any change broadly, and continuing to monitor non-significant variants rather than discarding them outright.

---
*Report generated by the Dynamic A/B Test Interpreter pipeline.*

## Notes / next steps
- The mock LLM client uses keyword/dtype heuristics as a stand-in — swap in `GroqLLMClient` (fill in the commented API call) to have a real model read the schema and write the narrative instead.
- Currently only two-tailed tests are run, and every variant is compared against one baseline (no variant-vs-variant comparisons, e.g. 5% discount vs. 10% discount) — a natural v2 extension.
- The schema inspector currently expects one metric per run — extending it to detect and test *multiple* metrics at once (e.g. both `retention_1` and `retention_7`) is a straightforward next step.
- The validator is still keyword-based; a v2 could use an LLM-as-judge to catch subtler overstatements.